In [2]:
import random
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
from google.colab import drive

drive.mount('/content/drive')


# =========================================================
# 1. ENV
# =========================================================

class TagMARLEnv:
    """
    1 Pacman vs 2 Ghosts
    Partial observability with 8-ray radar

    Reward:
      - Pacman: -1 if min Manhattan distance to any ghost <= 2 else 0
      - Ghost i: +1 if Manhattan distance to pacman <= 2 else 0

    No catch termination. Only max_steps terminates.

    Actor input:
      - Pacman: radar only (40)
      - Ghost : radar only (40)

    Critic input:
      - Pacman: radar + relative positions to both ghosts (40 + 4 = 44)
      - Ghost : radar + relative position to pacman (40 + 2 = 42)
    """

    ACTIONS = {
        0: (-1, 0),   # UP
        1: (1, 0),    # DOWN
        2: (0, -1),   # LEFT
        3: (0, 1),    # RIGHT
        4: (0, 0),    # STAY
    }

    RAYS = [
        (-1, 0),   # N
        (-1, 1),   # NE
        (0, 1),    # E
        (1, 1),    # SE
        (1, 0),    # S
        (1, -1),   # SW
        (0, -1),   # W
        (-1, -1),  # NW
    ]

    def __init__(self, max_steps=200, seed=42, min_start_dist=4, pacman_speed=2, ghost_speed=1):
        self.max_steps = max_steps
        self.rng = random.Random(seed)
        self.min_start_dist = min_start_dist
        self.pacman_speed = pacman_speed
        self.ghost_speed = ghost_speed

        self.raw_map = [
            "###########",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "#000000000#",
            "###########",
        ]
        self.raw_map = [
            "###########",
            "#000000000#",
            "#000#00000#",
            "#000#00#00#",
            "#0#0#00#00#",
            "#0#0#00#00#",
            "#0#0#00#00#",
            "#0#0#00#00#",
            "#0#0#00000#",
            "#000000000#",
            "###########",
        ]

        self.H = len(self.raw_map)
        self.W = len(self.raw_map[0])

        self.empty_cells = [
            (r, c)
            for r in range(self.H)
            for c in range(self.W)
            if self.raw_map[r][c] == "0"
        ]

        self.reset()

    def reset(self):
        self.grid = [list(row) for row in self.raw_map]

        while True:
            sampled_positions = self.rng.sample(self.empty_cells, 3)
            pacman = sampled_positions[0]
            ghosts = sampled_positions[1:]

            if all(self.manhattan(pacman, g) >= self.min_start_dist for g in ghosts):
                self.pacman = pacman
                self.ghosts = ghosts
                break

        self.steps = 0
        self.done = False
        return self.get_obs()

    def in_bounds(self, r, c):
        return 0 <= r < self.H and 0 <= c < self.W

    def move(self, pos, action):
        dr, dc = self.ACTIONS[action]
        nr, nc = pos[0] + dr, pos[1] + dc

        if not self.in_bounds(nr, nc):
            return pos
        if self.grid[nr][nc] == "#":
            return pos
        return (nr, nc)

    def manhattan(self, a, b):
        return abs(a[0] - b[0]) + abs(a[1] - b[1])

    # -------------------------
    # Pacman radar
    # per ray: [wall_dist, ghost_dist, ghost_dx, ghost_dy, ghost_mask]
    # total = 8 * 5 = 40
    # -------------------------
    def pacman_obs(self):
        px, py = self.pacman
        feat = []
        scale_hw = max(self.H, self.W)

        for dr, dc in self.RAYS:
            wall_dist = 0
            ghost_found = False
            ghost_dist = 0
            ghost_dx = 0
            ghost_dy = 0
            ghost_mask = 0

            r, c = px, py
            while True:
                r += dr
                c += dc
                wall_dist += 1

                if not self.in_bounds(r, c) or self.grid[r][c] == "#":
                    break

                if not ghost_found:
                    for g in self.ghosts:
                        if (r, c) == g:
                            ghost_found = True
                            ghost_dist = wall_dist
                            ghost_dx = g[0] - px
                            ghost_dy = g[1] - py
                            ghost_mask = 1
                            break

            feat.extend([
                wall_dist / scale_hw,
                ghost_dist / scale_hw,
                ghost_dx / self.H,
                ghost_dy / self.W,
                float(ghost_mask),
            ])

        return np.array(feat, dtype=np.float32)

    # -------------------------
    # Ghost radar
    # per ray: [wall_dist, pacman_dist, pacman_dx, pacman_dy, source]
    # source: 0 none, 1 self saw, 2 other ghost saw
    # total = 8 * 5 = 40
    # -------------------------
    def ghost_obs_single(self, idx):
        gx, gy = self.ghosts[idx]
        other_ghosts = [self.ghosts[j] for j in range(len(self.ghosts)) if j != idx]

        other_sees = any(self._ghost_can_see_pacman(og) for og in other_ghosts)
        feat = []
        scale_hw = max(self.H, self.W)

        for dr, dc in self.RAYS:
            wall_dist = 0
            pacman_found = False
            pacman_dist = 0
            pacman_dx = 0
            pacman_dy = 0
            source = 0

            r, c = gx, gy
            while True:
                r += dr
                c += dc
                wall_dist += 1

                if not self.in_bounds(r, c) or self.grid[r][c] == "#":
                    break

                if (r, c) == self.pacman and not pacman_found:
                    pacman_found = True
                    pacman_dist = wall_dist
                    pacman_dx = self.pacman[0] - gx
                    pacman_dy = self.pacman[1] - gy
                    source = 1
                    break

            if source == 0 and other_sees:
                pacman_dx = self.pacman[0] - gx
                pacman_dy = self.pacman[1] - gy
                pacman_dist = abs(pacman_dx) + abs(pacman_dy)
                source = 2

            feat.extend([
                wall_dist / scale_hw,
                pacman_dist / scale_hw,
                pacman_dx / self.H,
                pacman_dy / self.W,
                source / 2.0,
            ])

        return np.array(feat, dtype=np.float32)

    def _ghost_can_see_pacman(self, ghost_pos):
        gx, gy = ghost_pos
        for dr, dc in self.RAYS:
            r, c = gx, gy
            while True:
                r += dr
                c += dc
                if not self.in_bounds(r, c) or self.grid[r][c] == "#":
                    break
                if (r, c) == self.pacman:
                    return True
        return False

    def get_obs(self):
        pacman_actor = self.pacman_obs()  # [40]
        ghost_actor = np.stack(
            [self.ghost_obs_single(i) for i in range(len(self.ghosts))],
            axis=0
        )  # [G,40]

        px, py = self.pacman

        pacman_rel = []
        for gx, gy in self.ghosts:
            pacman_rel.extend([
                (gx - px) / self.H,
                (gy - py) / self.W,
            ])
        pacman_rel = np.array(pacman_rel, dtype=np.float32)  # [4]
        pacman_critic = np.concatenate([pacman_actor, pacman_rel], axis=0)  # [44]

        ghost_critic = []
        for i, (gx, gy) in enumerate(self.ghosts):
            rel = np.array([
                (px - gx) / self.H,
                (py - gy) / self.W,
            ], dtype=np.float32)  # [2]
            ghost_critic.append(np.concatenate([ghost_actor[i], rel], axis=0))  # [42]
        ghost_critic = np.stack(ghost_critic, axis=0)  # [G,42]

        return {
            "pacman_actor": pacman_actor,
            "pacman_critic": pacman_critic,
            "ghost_actor": ghost_actor,
            "ghost_critic": ghost_critic,
        }

    def step(self, pacman_action, ghost_actions):
        if self.done:
            raise RuntimeError("Episode ended. Call reset().")

        self.steps += 1

        for _ in range(self.pacman_speed):
            self.pacman = self.move(self.pacman, pacman_action)

        for _ in range(self.ghost_speed):
            self.ghosts = [self.move(g, a) for g, a in zip(self.ghosts, ghost_actions)]

        min_dist = min(self.manhattan(self.pacman, g) for g in self.ghosts)

        pacman_reward = -1.0 if min_dist <= 2 else 0.0
        ghost_rewards = np.array(
            [1.0 if self.manhattan(self.pacman, g) <= 2 else 0.0 for g in self.ghosts],
            dtype=np.float32
        )

        if self.steps >= self.max_steps:
            self.done = True

        obs = self.get_obs()
        rewards = {
            "pacman": pacman_reward,
            "ghosts": ghost_rewards
        }
        dones = {
            "pacman": float(self.done),
            "ghosts": np.array([float(self.done)] * len(self.ghosts), dtype=np.float32)
        }
        return obs, rewards, dones, {}

    def render_text(self):
        board = [row[:] for row in self.grid]
        pr, pc = self.pacman
        board[pr][pc] = "P"
        for i, (gr, gc) in enumerate(self.ghosts):
            board[gr][gc] = str(i + 1)
        print("\n".join("".join(row) for row in board))
        print(f"steps={self.steps}, done={self.done}")


# =========================================================
# 2. MODEL
# =========================================================

class RecurrentActorCritic(nn.Module):
    def __init__(self, actor_obs_dim, critic_obs_dim, action_dim, hidden_dim=128):
        super().__init__()
        self.hidden_dim = hidden_dim

        self.actor_encoder = nn.Sequential(
            nn.Linear(actor_obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        self.critic_encoder = nn.Sequential(
            nn.Linear(critic_obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )

        self.actor_lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=False)
        self.critic_lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=False)

        self.policy_head = nn.Linear(hidden_dim, action_dim)
        self.value_head = nn.Linear(hidden_dim, 1)

    def init_hidden(self, batch_size, device):
        actor_h = torch.zeros(1, batch_size, self.hidden_dim, device=device)
        actor_c = torch.zeros(1, batch_size, self.hidden_dim, device=device)
        critic_h = torch.zeros(1, batch_size, self.hidden_dim, device=device)
        critic_c = torch.zeros(1, batch_size, self.hidden_dim, device=device)
        return actor_h, actor_c, critic_h, critic_c

    def forward_step(self, actor_obs, critic_obs, hidden):
        actor_h, actor_c, critic_h, critic_c = hidden

        actor_x = self.actor_encoder(actor_obs).unsqueeze(0)
        critic_x = self.critic_encoder(critic_obs).unsqueeze(0)

        actor_out, (actor_h, actor_c) = self.actor_lstm(actor_x, (actor_h, actor_c))
        critic_out, (critic_h, critic_c) = self.critic_lstm(critic_x, (critic_h, critic_c))

        actor_out = actor_out.squeeze(0)
        critic_out = critic_out.squeeze(0)

        logits = self.policy_head(actor_out)
        value = self.value_head(critic_out).squeeze(-1)

        hidden = (actor_h, actor_c, critic_h, critic_c)
        return logits, value, hidden

    def forward_sequence(self, actor_obs_seq, critic_obs_seq, done_seq, init_hidden):
        T, B, _ = actor_obs_seq.shape
        actor_h, actor_c, critic_h, critic_c = init_hidden

        logits_list = []
        values_list = []

        for t in range(T):
            if t > 0:
                mask = (1.0 - done_seq[t - 1]).view(1, B, 1)
                actor_h = actor_h * mask
                actor_c = actor_c * mask
                critic_h = critic_h * mask
                critic_c = critic_c * mask

            actor_x = self.actor_encoder(actor_obs_seq[t]).unsqueeze(0)
            critic_x = self.critic_encoder(critic_obs_seq[t]).unsqueeze(0)

            actor_out, (actor_h, actor_c) = self.actor_lstm(actor_x, (actor_h, actor_c))
            critic_out, (critic_h, critic_c) = self.critic_lstm(critic_x, (critic_h, critic_c))

            actor_out = actor_out.squeeze(0)
            critic_out = critic_out.squeeze(0)

            logits = self.policy_head(actor_out)
            value = self.value_head(critic_out).squeeze(-1)

            logits_list.append(logits)
            values_list.append(value)

        hidden = (actor_h, actor_c, critic_h, critic_c)
        return torch.stack(logits_list, dim=0), torch.stack(values_list, dim=0), hidden


# =========================================================
# 3. BUFFER
# =========================================================

@dataclass
class RolloutBatch:
    actor_obs: torch.Tensor
    critic_obs: torch.Tensor
    actions: torch.Tensor
    logprobs: torch.Tensor
    rewards: torch.Tensor
    dones: torch.Tensor
    values: torch.Tensor
    advantages: torch.Tensor
    returns: torch.Tensor
    init_actor_h: torch.Tensor
    init_actor_c: torch.Tensor
    init_critic_h: torch.Tensor
    init_critic_c: torch.Tensor


def compute_gae(rewards, values, dones, gamma=0.99, lam=0.95):
    T, B = rewards.shape
    advantages = torch.zeros_like(rewards)
    last_adv = torch.zeros(B, device=rewards.device)
    next_value = torch.zeros(B, device=rewards.device)

    for t in reversed(range(T)):
        mask = 1.0 - dones[t]
        delta = rewards[t] + gamma * next_value * mask - values[t]
        last_adv = delta + gamma * lam * mask * last_adv
        advantages[t] = last_adv
        next_value = values[t]

    returns = advantages + values
    return advantages, returns


# =========================================================
# 4. PARALLEL ROLLOUT
# =========================================================

def collect_rollout_parallel(envs, pacman_net, ghost_net, horizon, device):
    num_envs = len(envs)
    num_ghosts = len(envs[0].ghosts)

    obs = [env.reset() for env in envs]

    pac_hidden = pacman_net.init_hidden(num_envs, device)
    ghost_hidden = ghost_net.init_hidden(num_envs * num_ghosts, device)

    pac_init = tuple(x.detach().clone() for x in pac_hidden)
    ghost_init = tuple(x.detach().clone() for x in ghost_hidden)

    pac_actor_obs_buf, pac_critic_obs_buf = [], []
    pac_act_buf, pac_logp_buf = [], []
    pac_rew_buf, pac_done_buf, pac_val_buf = [], [], []

    ghost_actor_obs_buf, ghost_critic_obs_buf = [], []
    ghost_act_buf, ghost_logp_buf = [], []
    ghost_rew_buf, ghost_done_buf, ghost_val_buf = [], [], []

    for _ in range(horizon):
        pac_actor_obs = torch.tensor(
            np.stack([o["pacman_actor"] for o in obs]),
            dtype=torch.float32, device=device
        )

        pac_critic_obs = torch.tensor(
            np.stack([o["pacman_critic"] for o in obs]),
            dtype=torch.float32, device=device
        )

        ghost_actor_obs = torch.tensor(
            np.concatenate([o["ghost_actor"] for o in obs], axis=0),
            dtype=torch.float32, device=device
        )

        ghost_critic_obs = torch.tensor(
            np.concatenate([o["ghost_critic"] for o in obs], axis=0),
            dtype=torch.float32, device=device
        )

        with torch.no_grad():
            pac_logits, pac_values, pac_hidden = pacman_net.forward_step(
                pac_actor_obs, pac_critic_obs, pac_hidden
            )
            pac_dist = Categorical(logits=pac_logits)
            pac_actions = pac_dist.sample()
            pac_logprobs = pac_dist.log_prob(pac_actions)

            ghost_logits, ghost_values, ghost_hidden = ghost_net.forward_step(
                ghost_actor_obs, ghost_critic_obs, ghost_hidden
            )
            ghost_dist = Categorical(logits=ghost_logits)
            ghost_actions = ghost_dist.sample()
            ghost_logprobs = ghost_dist.log_prob(ghost_actions)

        ghost_actions_np = ghost_actions.view(num_envs, num_ghosts).cpu().numpy()

        next_obs = []
        pac_rewards = []
        pac_dones = []
        ghost_rewards = []
        ghost_dones = []

        for env_i, env in enumerate(envs):
            o, r, d, _ = env.step(
                int(pac_actions[env_i].item()),
                ghost_actions_np[env_i].tolist()
            )

            next_obs.append(o)
            pac_rewards.append(r["pacman"])
            pac_dones.append(float(d["pacman"]))
            ghost_rewards.append(r["ghosts"])
            ghost_dones.append(d["ghosts"])

        pac_done_tensor = torch.tensor(pac_dones, dtype=torch.float32, device=device)
        ghost_done_tensor = torch.tensor(
            np.concatenate(ghost_dones), dtype=torch.float32, device=device
        )

        pac_actor_obs_buf.append(pac_actor_obs)
        pac_critic_obs_buf.append(pac_critic_obs)
        pac_act_buf.append(pac_actions)
        pac_logp_buf.append(pac_logprobs)
        pac_val_buf.append(pac_values)
        pac_rew_buf.append(torch.tensor(pac_rewards, dtype=torch.float32, device=device))
        pac_done_buf.append(pac_done_tensor)

        ghost_actor_obs_buf.append(ghost_actor_obs)
        ghost_critic_obs_buf.append(ghost_critic_obs)
        ghost_act_buf.append(ghost_actions)
        ghost_logp_buf.append(ghost_logprobs)
        ghost_val_buf.append(ghost_values)
        ghost_rew_buf.append(torch.tensor(np.concatenate(ghost_rewards), dtype=torch.float32, device=device))
        ghost_done_buf.append(ghost_done_tensor)

        for env_i, done_flag in enumerate(pac_dones):
            if done_flag:
                next_obs[env_i] = envs[env_i].reset()

        pac_mask = (1.0 - pac_done_tensor).view(1, num_envs, 1)
        pac_hidden = tuple(h * pac_mask for h in pac_hidden)

        ghost_mask = (1.0 - ghost_done_tensor).view(1, num_envs * num_ghosts, 1)
        ghost_hidden = tuple(h * ghost_mask for h in ghost_hidden)

        obs = next_obs

    pac_actor_obs = torch.stack(pac_actor_obs_buf, dim=0)
    pac_critic_obs = torch.stack(pac_critic_obs_buf, dim=0)
    pac_actions = torch.stack(pac_act_buf, dim=0)
    pac_logprobs = torch.stack(pac_logp_buf, dim=0)
    pac_values = torch.stack(pac_val_buf, dim=0)
    pac_rewards = torch.stack(pac_rew_buf, dim=0)
    pac_dones = torch.stack(pac_done_buf, dim=0)

    ghost_actor_obs = torch.stack(ghost_actor_obs_buf, dim=0)
    ghost_critic_obs = torch.stack(ghost_critic_obs_buf, dim=0)
    ghost_actions = torch.stack(ghost_act_buf, dim=0)
    ghost_logprobs = torch.stack(ghost_logp_buf, dim=0)
    ghost_values = torch.stack(ghost_val_buf, dim=0)
    ghost_rewards = torch.stack(ghost_rew_buf, dim=0)
    ghost_dones = torch.stack(ghost_done_buf, dim=0)

    pac_adv, pac_ret = compute_gae(pac_rewards, pac_values, pac_dones)
    ghost_adv, ghost_ret = compute_gae(ghost_rewards, ghost_values, ghost_dones)

    pac_batch = RolloutBatch(
        actor_obs=pac_actor_obs,
        critic_obs=pac_critic_obs,
        actions=pac_actions,
        logprobs=pac_logprobs,
        rewards=pac_rewards,
        dones=pac_dones,
        values=pac_values,
        advantages=pac_adv,
        returns=pac_ret,
        init_actor_h=pac_init[0],
        init_actor_c=pac_init[1],
        init_critic_h=pac_init[2],
        init_critic_c=pac_init[3],
    )

    ghost_batch = RolloutBatch(
        actor_obs=ghost_actor_obs,
        critic_obs=ghost_critic_obs,
        actions=ghost_actions,
        logprobs=ghost_logprobs,
        rewards=ghost_rewards,
        dones=ghost_dones,
        values=ghost_values,
        advantages=ghost_adv,
        returns=ghost_ret,
        init_actor_h=ghost_init[0],
        init_actor_c=ghost_init[1],
        init_critic_h=ghost_init[2],
        init_critic_c=ghost_init[3],
    )

    return pac_batch, ghost_batch


# =========================================================
# 5. PPO UPDATE
# =========================================================

def ppo_update(
    net,
    optimizer,
    batch,
    clip_eps=0.2,
    vf_coef=0.5,
    ent_coef=0.01,
    epochs=4,
    max_grad_norm=0.5,
):
    actor_obs = batch.actor_obs
    critic_obs = batch.critic_obs
    actions = batch.actions
    old_logprobs = batch.logprobs.detach()
    dones = batch.dones
    advantages = batch.advantages.detach()
    returns = batch.returns.detach()

    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    init_hidden = (
        batch.init_actor_h.detach(),
        batch.init_actor_c.detach(),
        batch.init_critic_h.detach(),
        batch.init_critic_c.detach(),
    )

    for _ in range(epochs):
        logits, values, _ = net.forward_sequence(actor_obs, critic_obs, dones, init_hidden)
        dist = Categorical(logits=logits)

        new_logprobs = dist.log_prob(actions)
        entropy = dist.entropy().mean()

        ratio = torch.exp(new_logprobs - old_logprobs)
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * advantages

        policy_loss = -torch.min(surr1, surr2).mean()
        value_loss = ((values - returns) ** 2).mean()
        loss = policy_loss + vf_coef * value_loss - ent_coef * entropy

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(net.parameters(), max_grad_norm)
        optimizer.step()


# =========================================================
# 6. TRAIN
# =========================================================

def load_training_checkpoint(path, pacman_net, ghost_net, pac_optimizer, ghost_optimizer, device):
    ckpt = torch.load(path, map_location=device)

    pacman_net.load_state_dict(ckpt["pacman_model_state_dict"])
    ghost_net.load_state_dict(ckpt["ghost_model_state_dict"])
    pac_optimizer.load_state_dict(ckpt["pacman_optimizer_state_dict"])
    ghost_optimizer.load_state_dict(ckpt["ghost_optimizer_state_dict"])

    stats = ckpt.get("stats", {
        "pacman_return": [],
        "ghost_return": [],
    })
    start_update = ckpt.get("update", 0) + 1

    print(f"[Resume Loaded] from update={ckpt.get('update', 0)}")
    return start_update, stats


def save_checkpoint(save_dir, update, pacman_net, ghost_net, pac_optimizer, ghost_optimizer, stats):
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    ckpt_path = save_dir / f"checkpoint_update_{update}.pt"

    torch.save({
        "update": update,
        "pacman_model_state_dict": pacman_net.state_dict(),
        "ghost_model_state_dict": ghost_net.state_dict(),
        "pacman_optimizer_state_dict": pac_optimizer.state_dict(),
        "ghost_optimizer_state_dict": ghost_optimizer.state_dict(),
        "stats": stats,
    }, ckpt_path)

    print(f"[Checkpoint Saved] {ckpt_path}")


def train_marl(
    num_envs=8,
    total_updates=200,
    horizon=128,
    lr=3e-4,
    hidden_dim=128,
    seed=42,
    max_steps=200,
    device=None,
    save_every=50,
    save_dir="/content/drive/MyDrive/tag_marl_checkpoints",
    resume_path=None,
):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    envs = [TagMARLEnv(max_steps=max_steps, seed=seed + i) for i in range(num_envs)]

    pacman_net = RecurrentActorCritic(
        actor_obs_dim=40,
        critic_obs_dim=44,
        action_dim=5,
        hidden_dim=hidden_dim
    ).to(device)

    ghost_net = RecurrentActorCritic(
        actor_obs_dim=40,
        critic_obs_dim=42,
        action_dim=5,
        hidden_dim=hidden_dim
    ).to(device)

    pac_optimizer = optim.Adam(pacman_net.parameters(), lr=lr)
    ghost_optimizer = optim.Adam(ghost_net.parameters(), lr=lr)

    stats = {
        "pacman_return": [],
        "ghost_return": [],
    }

    start_update = 1

    if resume_path is not None:
        start_update, stats = load_training_checkpoint(
            path=resume_path,
            pacman_net=pacman_net,
            ghost_net=ghost_net,
            pac_optimizer=pac_optimizer,
            ghost_optimizer=ghost_optimizer,
            device=device,
        )

    for update in range(start_update, total_updates + 1):
        pac_batch, ghost_batch = collect_rollout_parallel(
            envs=envs,
            pacman_net=pacman_net,
            ghost_net=ghost_net,
            horizon=horizon,
            device=device,
        )

        ppo_update(pacman_net, pac_optimizer, pac_batch)
        ppo_update(ghost_net, ghost_optimizer, ghost_batch)

        pac_return = pac_batch.rewards.sum(dim=0).mean().item()
        ghost_return = ghost_batch.rewards.sum(dim=0).mean().item()

        stats["pacman_return"].append(pac_return)
        stats["ghost_return"].append(ghost_return)

        if update % 10 == 0:
            print(
                f"[{update:4d}/{total_updates}] "
                f"PacmanMeanReturn={pac_return:8.3f} | "
                f"GhostMeanReturn={ghost_return:8.3f}"
            )

        if save_every is not None and update % save_every == 0:
            save_checkpoint(
                save_dir=save_dir,
                update=update,
                pacman_net=pacman_net,
                ghost_net=ghost_net,
                pac_optimizer=pac_optimizer,
                ghost_optimizer=ghost_optimizer,
                stats=stats,
            )

    if save_every is not None and total_updates % save_every != 0:
        save_checkpoint(
            save_dir=save_dir,
            update=total_updates,
            pacman_net=pacman_net,
            ghost_net=ghost_net,
            pac_optimizer=pac_optimizer,
            ghost_optimizer=ghost_optimizer,
            stats=stats,
        )

    return envs[0], pacman_net, ghost_net, stats


# =========================================================
# 7. EVAL
# =========================================================

@torch.no_grad()
def evaluate(env, pacman_net, ghost_net, episodes=3, device=None, render=False):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    pacman_net.eval()
    ghost_net.eval()

    for ep in range(episodes):
        obs = env.reset()
        done = False

        pac_hidden = pacman_net.init_hidden(1, device)
        ghost_hidden = ghost_net.init_hidden(len(env.ghosts), device)

        pac_total = 0.0
        ghost_total = np.zeros(len(env.ghosts), dtype=np.float32)

        while not done:
            pac_actor_obs = torch.tensor(obs["pacman_actor"], dtype=torch.float32, device=device).unsqueeze(0)
            pac_critic_obs = torch.tensor(obs["pacman_critic"], dtype=torch.float32, device=device).unsqueeze(0)

            ghost_actor_obs = torch.tensor(obs["ghost_actor"], dtype=torch.float32, device=device)
            ghost_critic_obs = torch.tensor(obs["ghost_critic"], dtype=torch.float32, device=device)

            pac_logits, _, pac_hidden = pacman_net.forward_step(
                pac_actor_obs, pac_critic_obs, pac_hidden
            )
            ghost_logits, _, ghost_hidden = ghost_net.forward_step(
                ghost_actor_obs, ghost_critic_obs, ghost_hidden
            )

            pac_dist = Categorical(logits=pac_logits)
            pac_action = pac_dist.sample().item()

            ghost_dist = Categorical(logits=ghost_logits)
            ghost_actions = ghost_dist.sample().cpu().numpy().tolist()

            obs, rewards, dones, _ = env.step(pac_action, ghost_actions)
            pac_total += rewards["pacman"]
            ghost_total += rewards["ghosts"]
            done = bool(dones["pacman"])

            if render:
                env.render_text()
                print("pac_reward:", rewards["pacman"], "ghost_rewards:", rewards["ghosts"])
                print("-" * 40)

        print(f"[Eval {ep+1}] Pacman={pac_total:.2f}, GhostMean={ghost_total.mean():.2f}")

Mounted at /content/drive


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

env, pacman_net, ghost_net, stats = train_marl(
    num_envs=8,
    total_updates=1000,
    horizon=128,
    lr=3e-4,
    hidden_dim=128,
    seed=42,
    max_steps=200,
    device=device,
    save_every=20,
    save_dir="/content/drive/MyDrive/tag_marl_checkpoints",
    #resume_path="/content/drive/MyDrive/tag_marl_checkpoints/checkpoint_update_380.pt",
)

evaluate(env, pacman_net, ghost_net, episodes=3, device=device, render=True)

[  10/1000] PacmanMeanReturn= -15.750 | GhostMeanReturn=   8.000
[  20/1000] PacmanMeanReturn= -27.250 | GhostMeanReturn=  14.250
[Checkpoint Saved] /content/drive/MyDrive/tag_marl_checkpoints/checkpoint_update_20.pt
[  30/1000] PacmanMeanReturn= -30.250 | GhostMeanReturn=  15.812
[  40/1000] PacmanMeanReturn= -10.500 | GhostMeanReturn=   5.375
[Checkpoint Saved] /content/drive/MyDrive/tag_marl_checkpoints/checkpoint_update_40.pt
[  50/1000] PacmanMeanReturn= -45.750 | GhostMeanReturn=  23.875
[  60/1000] PacmanMeanReturn= -44.125 | GhostMeanReturn=  34.312
[Checkpoint Saved] /content/drive/MyDrive/tag_marl_checkpoints/checkpoint_update_60.pt
[  70/1000] PacmanMeanReturn=  -9.250 | GhostMeanReturn=   5.000
[  80/1000] PacmanMeanReturn= -24.625 | GhostMeanReturn=  13.438
[Checkpoint Saved] /content/drive/MyDrive/tag_marl_checkpoints/checkpoint_update_80.pt
[  90/1000] PacmanMeanReturn=  -9.250 | GhostMeanReturn=   5.312
[ 100/1000] PacmanMeanReturn= -58.500 | GhostMeanReturn=  43.812
[C

KeyboardInterrupt: 